# Entrenamiento de clasificador de imágenes médicas

Para entrenar un clasificador de imagenes desde cero que aprenda sin sobreajuste, se requiere el uso de millones de imagenes.
Por lo que es mas practico utilizar un modelo pre-entrenado con algun dominio general y ajustar los pesos al problema especifico a evaluar.

Dentro de este notebook, vamos a entrenar una arquitectura ResNet34 con imágenes de una base de datos de imagenes médicas estandarizadas, para aprender a como entrenar y evaluar un clasificador con una Red Neuronal Profunda.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision.transforms import v2 as transforms
from torchvision.models import resnet34
from torchvision.ops import sigmoid_focal_loss

from torchmetrics.classification import (
    MultilabelAccuracy,
    MultilabelF1Score,
    MultilabelAUROC,
)
from sklearn import metrics

from tqdm import tqdm
from medmnist import ChestMNIST

datafolder = "datasets/medmnist"

revisamos GPU disponible

In [ ]:
# revisar GPU disponible

if torch.cuda.is_available():
    str_device = "cuda"
elif torch.backends.mps.is_available():
    str_device = "mps"
elif torch.xpu.is_available():
    str_device = "xpu"
else:
    str_device = "cpu"

print(f"Acelerador {str_device} disponible")
device = torch.device(str_device)

In [ ]:
# parametros de entrenamiento
lr = 1e-3
batch_size = 32
epochs = 10
image_size = (224, 224)
n_labels = 14  # numero de etiquetas en el dataset


## Dataset medMNIST

El dataset [medMNIST](https://github.com/MedMNIST/MedMNIST) es un conjunto de datasets de prueba, estandarizados, enfocados en problemas de clasificación de imágenes biomedicas en 2D y 3D.

Dentro de sus caracteristicas, posee versiones de imagenes para ejemplos pequeños `28x28`, hasta `224x224`
Imagenes 2D y 3D 

Posee ejemplos de tejido de colon con patologías, Rayos X de pecho, Dermatoscopia, Retina, Fondo de ojo, Analisis de celulas sanguineas, imagenes de tomografía abdominal, etc.

### ChestMNIST
Este subconjunto utiliza la base de datos `ChestX-Ray14`, la que contiene alrededor de 112120 imagenes de rayos X frontales. con 30805 pacientes.
Las clases presentes en este conjunto son: 
Atelectasis, Cardiomegalia, Efusión, Infiltración, Masa, Nodulo, Neumonia, Neumotorax, Consolidación, Edema, Enfisema, Fibrosis, Engrosamiento Pleural, Hernia|

In [ ]:
label_names = [
    "Atelectasis",
    "Cardiomegalia",
    "Efusión",
    "Infiltración",
    "Masa",
    "Nodulo",
    "Neumonia",
    "Neumotorax",
    "Consolidación",
    "Edema",
    "Enfisema",
    "Fibrosis",
    "Engrosamiento Pleural",
    "Hernia",
]

In [ ]:
train_transforms = transforms.Compose(
    [
        transforms.ToImage(),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.ToDtype(torch.float32, scale=True),
    ]
)

test_transforms = transforms.Compose(
    [
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
    ]
)

In [ ]:
# Cargamos los datos
# Como ya descargamos el dataset, lo cargamos desde el disco
train_dataset = ChestMNIST(
    split="train",
    root=datafolder,
    download=False,
    size=224,
    as_rgb=True,
    transform=train_transforms,
)
val_dataset = ChestMNIST(
    split="val",
    root=datafolder,
    download=False,
    size=224,
    as_rgb=True,
    transform=test_transforms,
)
test_dataset = ChestMNIST(
    split="test",
    root=datafolder,
    download=False,
    size=224,
    as_rgb=True,
    transform=test_transforms,
)

Vamos a visualizar un ejemplo del conjunto de validación.
Como la imagen es ahora un tensor, de tamaño `(canales, ancho, largo)`, es necesario transponer la matriz y volverla un array de numpy para poder presentarla.

In [ ]:
imagen, label = val_dataset[24]
imagen = imagen.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(imagen)
present_labels = [label_names[l] for l in np.nonzero(label)[0]]

if present_labels:
    ax.set_title(", ".join(present_labels))
else:
    ax.set_title("No Label")
ax.axis("off")
plt.show()

Vamos a visualizar como esta distribuido este conjunto. El conjunto ChestMNIST contiene una o varias etiquetas de patologias presentes a nivel pulmonar.
Las imagenes sanas presentan en su vector de etiqueta solo 0. Por lo que los contaremos de forma separada.

In [ ]:
train_labels = train_dataset.labels

conteos = train_labels.sum(axis=0)
sanos = sum(train_labels.sum(axis=1) == 0)

# grafico
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
idx_clases = np.arange(len(conteos) + 1)
ax.bar(idx_clases, height=np.append(sanos, conteos))
ax.set_title("Distribución de clases en Entrenamieto")
ax.set_xlabel("Clase")
ax.set_ylabel("Frecuencia")
ax.set_xticks(
    idx_clases, labels=["Sano"] + label_names, rotation=60, va="top", ha="center"
)

## Dataloader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Modelo y optimizador

In [ ]:
modelo = resnet34(weights="DEFAULT")
modelo.fc = nn.Linear(modelo.fc.in_features, n_labels)
modelo = modelo.to(device)

In [ ]:
optimizer = optim.Adam(modelo.parameters(), lr=lr)
criterion = sigmoid_focal_loss

## Funciones de apoyo

In [ ]:
def train_epoch(modelo, dataloader, optimizer, criterion, device="cpu"):
    modelo.train()
    total_loss = 0.0

    acc_metric = MultilabelAccuracy(num_labels=n_labels, average="micro").to(device)
    f1_metric = MultilabelF1Score(num_labels=n_labels, average="micro").to(device)
    auroc_metric = MultilabelAUROC(num_labels=n_labels, average="micro").to(device)

    with tqdm(total=len(dataloader), desc="Entrenando") as pbar:
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = modelo(images)
            loss = criterion(outputs, labels.float(), alpha=0.3, reduction="mean")

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            acc_metric.update(outputs, labels)
            f1_metric.update(outputs, labels)
            auroc_metric.update(outputs, labels)

            # computamos para barra de progreso
            acc = acc_metric.compute()
            f1 = f1_metric.compute()
            auroc = auroc_metric.compute()
            pbar.set_postfix_str(
                f"Loss: {total_loss / (pbar.n + 1):.4f}, Acc: {acc:.4f}, F1: {f1:.4f}, AUROC: {auroc:.4f}"
            )
            pbar.update(1)

    return total_loss / len(dataloader)


In [ ]:
def eval_epoch(modelo, dataloader, criterion, device="cpu"):
    modelo.eval()
    total_loss = 0.0

    acc_metric = MultilabelAccuracy(num_labels=n_labels, average="micro").to(device)
    f1_metric = MultilabelF1Score(num_labels=n_labels, average="micro").to(device)
    auroc_metric = MultilabelAUROC(num_labels=n_labels, average="micro").to(device)

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluando"):
            images, labels = images.to(device), labels.to(device)

            outputs = modelo(images)
            loss = criterion(outputs, labels.float(), alpha=0.3, reduction="mean")

            total_loss += loss.item()
            acc_metric.update(outputs, labels)
            f1_metric.update(outputs, labels)
            auroc_metric.update(outputs, labels)

    acc = acc_metric.compute()
    f1 = f1_metric.compute()
    auroc = auroc_metric.compute()

    print(
        f"Eval Loss: {total_loss / len(dataloader):.4f}, Acc: {acc:.4f}, F1: {f1:.4f}, AUROC: {auroc:.4f}"
    )
    return total_loss / len(dataloader)

## Ciclo de entrenamiento

In [ ]:
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    train_loss = train_epoch(modelo, train_loader, optimizer, criterion, device)
    val_loss = eval_epoch(modelo, val_loader, criterion, device)

In [ ]:
torch.save(modelo.state_dict(), "saves/modelo_medmnist.pth")

## Testing (WIP)